# Baseline TF-IDF — without 'SOAP / Chart / Progress Notes' specialty
**Experiment:** removing `SOAP / Chart / Progress Notes` chunks improve TF-IDF retrieval metrics?

In [7]:
import json
import pickle
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.retrieval import TFIDFRetriever, evaluate_queries

pd.set_option("display.max_colwidth", 80)

In [8]:
with open(ROOT / "data" / "chunks.pkl", "rb") as f:
    chunks_all = pickle.load(f)

with open(ROOT / "data" / "test_queries.json") as f:
    test_queries = json.load(f)

EXCLUDED = {"SOAP / Chart / Progress Notes"}
chunks_filtered = [c for c in chunks_all if c["metadata"]["medical_specialty"] not in EXCLUDED]

removed = len(chunks_all) - len(chunks_filtered)
print(f"Original chunks:  {len(chunks_all):,}")
print(f"Filtered chunks:  {len(chunks_filtered):,}")
print(f"Removed:          {removed:,} ({removed / len(chunks_all):.1%})")

Original chunks:  38,838
Filtered chunks:  37,871
Removed:          967 (2.5%)


## Evaluation on 30 test queries

In [15]:
retriever_all = TFIDFRetriever(chunks_all, k=3)
retriever_filtered = TFIDFRetriever(chunks_filtered, k=3)
results_all = evaluate_queries(retriever_all, test_queries, k=3)
results_filtered = evaluate_queries(retriever_filtered, test_queries, k=3)

def summarize(df: pd.DataFrame, label: str) -> pd.DataFrame:
    by_cat = df.groupby("category")[["precision_at_k", "hit_at_k", "mrr"]].mean().round(3)
    overall = df[["precision_at_k", "hit_at_k", "mrr"]].mean().round(3).to_frame().T
    overall.index = ["OVERALL"]
    out = pd.concat([by_cat, overall])
    out.columns = pd.MultiIndex.from_product([[label], out.columns])
    return out

summary_all = summarize(results_all, "with SOAP")
summary_filtered = summarize(results_filtered, "without SOAP")
comparison = pd.concat([summary_all, summary_filtered], axis=1)
comparison

with SOAP                  without SOAP                
        precision_at_k hit_at_k   mrr precision_at_k hit_at_k    mrr
complex          0.300      0.6  0.55          0.300    0.700  0.433
direct           0.367      0.8  0.45          0.367    0.800  0.483
synonym          0.033      0.1  0.05          0.033    0.100  0.050
OVERALL          0.233      0.5  0.35          0.233    0.533  0.322

## Per-query retrieved specialties

In [12]:
def top_specialties(retriever, query, k=3):
    results = retriever.retrieve(query, k=k)
    return " | ".join(f"{r['metadata']['medical_specialty']} ({r['score']:.2f})" for r in results)

rows = []
for q in test_queries:
    rows.append({
        "id": q["id"],
        "category": q["category"],
        "query": q["query"],
        "expected": q["expected_specialty"],
        "with SOAP": top_specialties(retriever_all, q["query"]),
        "without SOAP": top_specialties(retriever_filtered, q["query"]),
    })

retrieved_df = pd.DataFrame(rows)
with pd.option_context("display.max_colwidth", 120, "display.width", 200):
    display(retrieved_df)

,id,category,query,expected,with SOAP,without SOAP
0,q01,direct,patient with chest pain and shortness of breath on exertion,Cardiovascular / Pulmonary,Consult - History and Phy. (0.38) | Cardiovascular / Pulmonary (0.38) | General Medicine (0.32),Consult - History and Phy. (0.38) | Cardiovascular / Pulmonary (0.38) | General Medicine (0.32)
1,q02,direct,right knee arthroscopy for torn meniscus,Orthopedic,Orthopedic (0.52) | Surgery (0.52) | Orthopedic (0.41),Surgery (0.52) | Orthopedic (0.52) | Surgery (0.41)
2,q03,direct,colonoscopy with biopsy for suspected polyps,Gastroenterology,Surgery (0.33) | Gastroenterology (0.33) | Surgery (0.28),Surgery (0.33) | Gastroenterology (0.33) | Surgery (0.28)
3,q04,direct,MRI of the lumbar spine with contrast,Radiology,Consult - History and Phy. (0.35) | Neurology (0.35) | Orthopedic (0.32),Neurology (0.35) | Consult - History and Phy. (0.35) | Orthopedic (0.32)
4,q05,direct,laparoscopic cholecystectomy for gallstones,Surgery,Discharge Summary (0.39) | Gastroenterology (0.39) | Gastroenterology (0.36),Gastroenterology (0.39) | Discharge Summary (0.39) | Consult - History and Phy. (0.36)
5,q06,direct,cystoscopy for recurrent urinary tract infections,Urology,Surgery (0.35) | Urology (0.35) | General Medicine (0.32),Urology (0.36) | Surgery (0.36) | Urology (0.33)
6,q07,direct,cesarean section delivery due to fetal distress,Obstetrics / Gynecology,Surgery (0.31) | Obstetrics / Gynecology (0.31) | Surgery (0.30),Surgery (0.30) | Obstetrics / Gynecology (0.30) | Obstetrics / Gynecology (0.30)
7,q08,direct,EEG for seizure disorder evaluation,Neurology,Consult - History and Phy. (0.39) | Neurology (0.39) | Psychiatry / Psychology (0.39),Psychiatry / Psychology (0.39) | Consult - History and Phy. (0.39) | Neurology (0.39)
8,q09,direct,tonsillectomy and adenoidectomy for chronic tonsillitis,ENT - Otolaryngology,Surgery (0.33) | ENT - Otolaryngology (0.33) | ENT - Otolaryngology (0.30),Surgery (0.33) | ENT - Otolaryngology (0.33) | Consult - History and Phy. (0.28)
9,q10,direct,bone marrow biopsy for suspected leukemia,Hematology - Oncology,SOAP / Chart / Progress Notes (0.38) | Hematology - Oncology (0.38) | Hematology - Oncology (0.27),Hematology - Oncology (0.31) | General Medicine (0.24) | Hematology - Oncology (0.24)


### Queries whose top-3 actually changed

In [13]:
changed = retrieved_df[retrieved_df["with SOAP"] != retrieved_df["without SOAP"]]
print(f"Queries with different top-3: {len(changed)} / {len(retrieved_df)}")
with pd.option_context("display.max_colwidth", 120, "display.width", 200):
    display(changed)

Queries with different top-3: 24 / 30


,id,category,query,expected,with SOAP,without SOAP
1,q02,direct,right knee arthroscopy for torn meniscus,Orthopedic,Orthopedic (0.52) | Surgery (0.52) | Orthopedic (0.41),Surgery (0.52) | Orthopedic (0.52) | Surgery (0.41)
3,q04,direct,MRI of the lumbar spine with contrast,Radiology,Consult - History and Phy. (0.35) | Neurology (0.35) | Orthopedic (0.32),Neurology (0.35) | Consult - History and Phy. (0.35) | Orthopedic (0.32)
4,q05,direct,laparoscopic cholecystectomy for gallstones,Surgery,Discharge Summary (0.39) | Gastroenterology (0.39) | Gastroenterology (0.36),Gastroenterology (0.39) | Discharge Summary (0.39) | Consult - History and Phy. (0.36)
5,q06,direct,cystoscopy for recurrent urinary tract infections,Urology,Surgery (0.35) | Urology (0.35) | General Medicine (0.32),Urology (0.36) | Surgery (0.36) | Urology (0.33)
6,q07,direct,cesarean section delivery due to fetal distress,Obstetrics / Gynecology,Surgery (0.31) | Obstetrics / Gynecology (0.31) | Surgery (0.30),Surgery (0.30) | Obstetrics / Gynecology (0.30) | Obstetrics / Gynecology (0.30)
7,q08,direct,EEG for seizure disorder evaluation,Neurology,Consult - History and Phy. (0.39) | Neurology (0.39) | Psychiatry / Psychology (0.39),Psychiatry / Psychology (0.39) | Consult - History and Phy. (0.39) | Neurology (0.39)
8,q09,direct,tonsillectomy and adenoidectomy for chronic tonsillitis,ENT - Otolaryngology,Surgery (0.33) | ENT - Otolaryngology (0.33) | ENT - Otolaryngology (0.30),Surgery (0.33) | ENT - Otolaryngology (0.33) | Consult - History and Phy. (0.28)
9,q10,direct,bone marrow biopsy for suspected leukemia,Hematology - Oncology,SOAP / Chart / Progress Notes (0.38) | Hematology - Oncology (0.38) | Hematology - Oncology (0.27),Hematology - Oncology (0.31) | General Medicine (0.24) | Hematology - Oncology (0.24)
10,q11,synonym,stomach ache and feeling sick after eating,Gastroenterology,Nephrology (0.17) | SOAP / Chart / Progress Notes (0.17) | SOAP / Chart / Progress Notes (0.16),Nephrology (0.18) | General Medicine (0.17) | Consult - History and Phy. (0.16)
11,q12,synonym,heart is racing and feels like it skips a beat,Cardiovascular / Pulmonary,Physical Medicine - Rehab (0.23) | SOAP / Chart / Progress Notes (0.23) | Obstetrics / Gynecology (0.23),Physical Medicine - Rehab (0.25) | Obstetrics / Gynecology (0.24) | Emergency Room Reports (0.22)
